## 7. Class Imbalance Handling--------------------------------------------------

# SMOTE

In [ ]:
# Import imbalance-handling libraries
# ImbPipeline is used instead of sklearn Pipeline so SMOTE only runs inside CV folds
# This prevents data leakage from the validation fold into training
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight

print("Imbalance handling libraries imported.")

Imbalance handling libraries imported.


DECISION TREE

In [ ]:
# SMOTE w Decision Tree
# SMOTE generates synthetic minority class samples to balance the training set
# Using imblearn Pipeline to ensures SMOTE is applied only inside each CV fold
smote_dt_pipe = ImbPipeline([
    ('preprocess', preprocess),
    ('smote', SMOTE(random_state=RANDOM_STATE)),   # applied inside CV folds only
    ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

smote_dt_params = {
    'smote__k_neighbors': [3, 5],                        # neighbors used to generate synthetic points
    'clf__max_depth': [10, 20, 30, None],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__criterion': ['gini', 'entropy'],
    'clf__class_weight': [None, 'balanced']
}

# RandomizedSearchCV is faster than GridSearchCV for large parameter spaces
smote_dt_search = RandomizedSearchCV(
    smote_dt_pipe, param_distributions=smote_dt_params,
    n_iter=20, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
smote_dt_search.fit(X_train, y_train)
smote_dt_best = smote_dt_search.best_estimator_

print('Best params (SMOTE + DT):', smote_dt_search.best_params_)
print(f'Best CV F1 Macro: {smote_dt_search.best_score_:.4f}')
sdt_metrics = evaluate('SMOTE + Decision Tree \u2014 VALIDATION', smote_dt_best, X_val, y_val)
results_table.append({'Model': 'SMOTE + Decision Tree', **sdt_metrics})

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params (SMOTE + DT): {'smote__k_neighbors': 5, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 2, 'clf__max_depth': None, 'clf__criterion': 'entropy', 'clf__class_weight': 'balanced'}
Best CV F1 Macro: 0.8808

=== SMOTE + Decision Tree — VALIDATION ===
              precision    recall  f1-score   support

           1      0.921     0.937     0.929     42368
           2      0.950     0.927     0.939     56660
           3      0.916     0.932     0.924      7151
           4      0.838     0.878     0.858       549
           5      0.764     0.858     0.808      1899
           6      0.843     0.872     0.857      3474
           7      0.928     0.957     0.942      4102

    accuracy                          0.929    116203
   macro avg      0.880     0.909     0.894    116203
weighted avg      0.930     0.929     0.929    116203

Accuracy         : 0.9290
Balanced Accuracy: 0.9086
F1 Macro         : 0.8938

RANDOM FOREST

In [ ]:
# SMOTE w Random Forest
# Combining SMOTE with Random Forest helps the ensemble learn from balanced classes
smote_rf_pipe = ImbPipeline([
    ('preprocess', preprocess),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

smote_rf_params = {
    'smote__k_neighbors': [3, 5],
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [10, 20, None],
    'clf__min_samples_split': [2, 5],
    'clf__min_samples_leaf': [1, 2],
    'clf__max_features': ['sqrt', 'log2']   # controls feature randomness per split
}

# tuning by RandomizedSearchCV and StratifiedKFold (with cross validation)
smote_rf_search = RandomizedSearchCV(
    smote_rf_pipe, param_distributions=smote_rf_params,
    n_iter=20, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
smote_rf_search.fit(X_train, y_train)
smote_rf_best = smote_rf_search.best_estimator_

print('Best params (SMOTE + RF):', smote_rf_search.best_params_)
print(f'Best CV F1 Macro: {smote_rf_search.best_score_:.4f}')
srf_metrics = evaluate('SMOTE + Random Forest \u2014 VALIDATION', smote_rf_best, X_val, y_val)
results_table.append({'Model': 'SMOTE + Random Forest', **srf_metrics})

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params (SMOTE + RF): {'smote__k_neighbors': 3, 'clf__n_estimators': 300, 'clf__min_samples_split': 2, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': None}
Best CV F1 Macro: 0.9209

=== SMOTE + Random Forest — VALIDATION ===
              precision    recall  f1-score   support

           1      0.959     0.941     0.950     42368
           2      0.955     0.959     0.957     56660
           3      0.937     0.958     0.947      7151
           4      0.881     0.907     0.894       549
           5      0.841     0.891     0.865      1899
           6      0.879     0.927     0.902      3474
           7      0.956     0.970     0.963      4102

    accuracy                          0.951    116203
   macro avg      0.916     0.936     0.926    116203
weighted avg      0.951     0.951     0.951    116203

Accuracy         : 0.9507
Balanced Accuracy: 0.9362
F1 Macro         : 0.9256

Per-C

XGBOOST

In [ ]:
# SMOTE w XGBoost
# XGBoost still requires 0-indexed labels (y_train_adj)
smote_xgb_pipe = ImbPipeline([
    ('preprocess', preprocess),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', XGBClassifier(
        objective='multi:softprob',
        num_class=num_classes,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

smote_xgb_params = {
    'smote__k_neighbors': [3, 5],
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [4, 6, 8],
    'clf__learning_rate': [0.05, 0.1, 0.2],
    'clf__subsample': [0.7, 0.8, 1.0],          # fraction of samples used per tree
    'clf__colsample_bytree': [0.7, 0.8, 1.0]    # fraction of features used per tree
}

# tuning with RandomizedSearchCV and StratifiedKFold
smote_xgb_search = RandomizedSearchCV(
    smote_xgb_pipe, param_distributions=smote_xgb_params,
    n_iter=20, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
smote_xgb_search.fit(X_train, y_train_adj)   # 0-indexed for XGBoost
smote_xgb_best = smote_xgb_search.best_estimator_

print('Best params (SMOTE + XGB):', smote_xgb_search.best_params_)
print(f'Best CV F1 Macro: {smote_xgb_search.best_score_:.4f}')
# offset=1 to add 1 back so predictions are 1-7
sxgb_metrics = evaluate('SMOTE + XGBoost \u2014 VALIDATION', smote_xgb_best, X_val, y_val_adj, offset=1)
results_table.append({'Model': 'SMOTE + XGBoost', **sxgb_metrics})

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params (SMOTE + XGB): {'smote__k_neighbors': 3, 'clf__subsample': 0.7, 'clf__n_estimators': 300, 'clf__max_depth': 8, 'clf__learning_rate': 0.2, 'clf__colsample_bytree': 1.0}
Best CV F1 Macro: 0.9230

=== SMOTE + XGBoost — VALIDATION ===
              precision    recall  f1-score   support

           0      0.000     0.000     0.000     42368
           1      0.057     0.042     0.048     56660
           2      0.001     0.008     0.002      7151
           3      0.004     0.056     0.008       549
           4      0.000     0.000     0.000      1899
           5      0.001     0.001     0.001      3474
           6      0.000     0.000     0.000      4102
           7      0.000     0.000     0.000         0

    accuracy                          0.021    116203
   macro avg      0.008     0.013     0.007    116203
weighted avg      0.028     0.021     0.024    116203

Accuracy         : 0.0211
Balanced Accuracy: